# Pertemuan 3: Pengkondisian dan Perulangan (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin

**Dataset (sekunder / publik):** [Stroke Prediction Dataset (Kaggle, fedesoriano)](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset), 5.110 baris × 12 kolom. Notebook ini berdiri sendiri dan memuat data langsung dari URL, jadi bisa diunggah ke Google Colab dan di-Run All tanpa upload file.

**Tujuan:** mengimplementasikan seluruh materi Pertemuan 3 (boolean, `if-elif-else`, `for`, `while`, `break`/`continue`, nested loop, comprehension) pada dataset kelompok, lalu menyaring data latih sebelum dipakai. Setiap tahapan ada kode, output, dan analisis di bawahnya untuk bahan screenshot laporan buku panduan.

## 1. Muat data

Sama seperti notebook `01`, data diambil langsung dari URL.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ray-project/raydp/master/tutorials/dataset/healthcare-dataset-stroke-data.csv"
df = pd.read_csv(URL)

print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])
df.head(3)

## 2. Boolean: pertanyaan True/False ke data

Materi slide 4. Operator perbandingan (`==, !=, >, <, >=, <=`), logika (`and, or, not`), keanggotaan (`in`), dan cek kosong (`is None`, `isna()`).

In [ ]:
# Contoh slide, tetap jalan
akurasi = 0.87
print(akurasi >= 0.80)  # True

label = "batik"
print(label in ["batik", "endek"])  # True

nilai = None
print(nilai is None)  # True

# Terapkan ke dataset
print("bmi kosong       :", int(df["bmi"].isna().sum()))
print("smoking Unknown  :", int((df["smoking_status"] == "Unknown").sum()))
print("bmi di atas 60   :", int((df["bmi"] > 60).sum()))

# Gabungan and / or / not pada satu baris
mask = (df["age"] >= 60) & (df["hypertension"] == 1) & (df["smoking_status"] != "Unknown")
print("lansia + hipertensi + smoking diketahui:", int(mask.sum()))

assert int(mask.sum()) > 0  # ponytail: cek kewarasan, bukan uji statistik

**Analisis:** mayoritas baris lolos cek tunggal, tetapi `smoking_status == "Unknown"` menempati sekitar 30% data. Itu missing value tersembunyi (temuan notebook `01`): bukan kategori bermakna, jadi pada tahap penyaringan nanti baris itu kita lewati dengan `continue`. Baris `bmi` kosong hanya sekitar 4%, dan BMI > 60 hanya belasan baris.

## 3. Percabangan `if-elif-else` dan urutan batas

Materi slide 5-6. Cabang pertama yang True yang jalan, jadi mulai dari batas paling ketat. Batas ikut dihitung pakai `>=`. Diuji tepat di bawah, pada, dan di atas batas.

In [ ]:
def kategori_glukosa(g):
    if g >= 200:      # paling ketat dulu
        return "sangat tinggi"
    elif g >= 140:
        return "tinggi"
    elif g >= 100:
        return "waspada"
    else:
        return "normal"


def kategori_risiko(age, hyp, heart):
    if heart == 1 and hyp == 1:
        return "tinggi"
    elif heart == 1 or hyp == 1:
        return "sedang"
    elif age >= 60:
        return "waspada"
    else:
        return "rendah"


# Uji batas: bawah, pas, atas
for g in [99.9, 100.0, 139.9, 140.0, 199.9, 200.0]:
    print(g, "->", kategori_glukosa(g))

print()
for i in range(5):
    r = df.iloc[i]
    print(
        f"baris {i}: glukosa={r['avg_glucose_level']:.1f} "
        f"({kategori_glukosa(r['avg_glucose_level'])}) "
        f"risiko={kategori_risiko(r['age'], r['hypertension'], r['heart_disease'])}"
    )

**Analisis:** urutan batas menentukan hasil. Kalau `>= 100` dicek duluan, nilai 250 ikut masuk cabang waspada dan cabang sangat tinggi tidak pernah tercapai. Pola yang sama dipakai di proyek: aturan ambang keputusan model juga sensitif terhadap urutan dan nilai batasnya. Fungsi `kategori_risiko` menunjukkan kombinasi `and` (keduanya harus ada) versus `or` (salah satu cukup).

## 4. `for`: telusuri koleksi yang sudah ada

Materi slide 7. Pakai `for` kalau item atau jumlah iterasi sudah tersedia: `enumerate`, `range`, `zip`.

In [ ]:
# enumerate: indeks + nilai
for indeks, skor in enumerate([78, 85, 91]):
    print(indeks, skor)

print()
# range: epoch ala slide
for epoch in range(1, 6):
    print("epoch", epoch)

print()
# enumerate + zip pada 5 pasien pertama
usia = df["age"].head(5).tolist()
bmi = df["bmi"].head(5).tolist()
for i, (u, b) in enumerate(zip(usia, bmi)):
    print(f"pasien {i}: usia={u} bmi={b}")

# for untuk agregasi sederhana, dicek melawan pandas
total, n = 0.0, 0
for v in df["age"].head(100):
    total += v
    n += 1
rata_loop = total / n
rata_pandas = df["age"].head(100).mean()
print("\nrata-rata usia 100 baris (loop):", round(rata_loop, 4))
assert abs(rata_loop - rata_pandas) < 1e-9

**Analisis:** `enumerate` memberi posisi baris (berguna saat melaporkan baris bermasalah ke laporan), `range` mengatur pengulangan bernomor seperti epoch, `zip` memasangkan dua kolom sejajar. Agregasi manual cocok dengan `mean()` pandas, jadi loop-nya benar. Untuk agregasi kolom penuh, pandas tetap yang dipakai di proyek karena jauh lebih cepat.

## 5. `while`: berhenti saat kondisi terpenuhi

Materi slide 8. Pakai `while` kalau titik berhenti dinamis: tetapkan kondisi awal, perbarui pengendali, pastikan ada syarat berhenti.

In [ ]:
# Contoh slide: loss menyusut tiap epoch
epoch, loss = 1, 1.0
while epoch <= 10 and loss > 0.10:
    loss *= 0.72
    print(epoch, round(loss, 3))
    epoch += 1  # tanpa baris ini loop tidak pernah berhenti

print()
# while pada dataset: kumpulkan 5 pasien berbmi valid pertama
ketemu, i = [], 0
while len(ketemu) < 5 and i < len(df):
    b = df.iloc[i]["bmi"]
    if pd.notna(b):
        ketemu.append((i, b))
    i += 1
print("5 bmi valid pertama (indeks, bmi):", ketemu)
assert len(ketemu) == 5

**Analisis:** loop `loss` berhenti di epoch 7 karena `loss` sudah di bawah 0,10 meski batas epoch 10 belum tercapai. Itulah bedanya dengan `for`: syarat berhenti ditentukan data, bukan jumlah yang tetap. Bentuk yang sama dipakai untuk pemindaian baris, selama variabel `i` selalu maju dan ada batas `len(df)` sebagai pengaman.

## 6. `break` dan `continue`

Materi slide 9. `continue` lewati satu iterasi (satu data diabaikan), `break` keluar total (kondisi fatal).

In [ ]:
# Contoh slide
for nilai in [12, None, 18, -3, 25]:
    if nilai is None:
        continue
    if nilai < 0:
        break
    print(nilai)

print()
# Versi dataset: pindai 20 bmi pertama
# Aturan demo: NaN dilewati (continue), bmi > 90 hentikan demo (break)
diproses, dilewati = 0, 0
for b in df["bmi"].head(20).tolist():
    if pd.isna(b):
        dilewati += 1
        continue
    if b > 90:
        print(f"bmi ekstrem {b}, hentikan pindaian demo")
        break
    diproses += 1
print(f"diproses={diproses} dilewati(NaN)={dilewati}")

**Analisis:** pada contoh slide yang tercetak hanya 12 dan 18. `None` dilewati, lalu `-3` menghentikan loop sehingga 25 tidak pernah diproses. Pada data nyata, `continue` adalah alat utama (lewati `bmi` kosong tanpa menggugurkan seluruh proses), sedangkan `break` hanya untuk interupsi darurat seperti pemeriksaan demo ini, bukan untuk filtering normal.

## 7. Nested loop: data bertingkat

Materi slide 10. Loop di dalam loop untuk kombinasi dua dimensi. Di sini: kelompok usia x hipertensi, hitung jumlah pasien dan proporsi stroke.

In [ ]:
def kelompok_usia(a):
    if a < 18:
        return "anak (<18)"
    elif a < 60:
        return "dewasa (18-59)"
    else:
        return "lansia (>=60)"


hasil = []
for ku in ["anak (<18)", "dewasa (18-59)", "lansia (>=60)"]:
    for hyp in [0, 1]:
        pot = df[(df["age"].apply(kelompok_usia) == ku) & (df["hypertension"] == hyp)]
        rate = pot["stroke"].mean() if len(pot) else 0.0
        hasil.append((ku, hyp, len(pot), round(float(rate), 4)))
        print(f"{ku} x hipertensi={hyp}: n={len(pot)} stroke_rate={rate:.4f}")

total = sum(h[2] for h in hasil)
print("\ntotal baris terpetakan:", total)
assert total == len(df)  # tiap baris masuk tepat satu sel

**Analisis:** enam sel mempartisi seluruh 5.110 baris tanpa sisa (assert total cocok). Polanya sesuai pengetahuan medis: proporsi stroke naik pada lansia dan pada penderita hipertensi. Nested loop di sini hanya 3 x 2 sel sehingga murah; untuk grid parameter besar, cara ini meledak kombinatorial dan sebaiknya diganti `groupby` atau pencarian terarah.

## 8. Comprehension: transformasi ringkas

Materi slide 11. Untuk ekspresi sederhana saja, bukan logika panjang. Kondisi di belakang.

In [ ]:
# Contoh slide
data = [10, None, 20, 30]
bersih = [x for x in data if x is not None]
skala = [x / 100 for x in bersih]
print(bersih, skala)

# Versi dataset: 10 bmi pertama
bmi_10 = df["bmi"].head(10).tolist()
bersih_bmi = [x for x in bmi_10 if pd.notna(x)]
skala_bmi = [x / 100 for x in bersih_bmi]
print("mentah :", bmi_10)
print("bersih :", bersih_bmi)
print("skala  :", [round(v, 4) for v in skala_bmi])
assert bersih_bmi == df["bmi"].head(10).dropna().tolist()

**Analisis:** satu baris comprehension menggantikan 3-4 baris loop saring, dan hasilnya identik dengan `dropna()` (assert cocok). Batasannya keterbacaan: kalau ada `if-elif-else` berlapis atau efek samping, tulis loop biasa saja.

## 9. Kasus ML: saring data latih sebelum dipakai

Materi slide 12. Satu alur: telusuri lalu validasi lalu perbaiki lalu simpan. Lewati fitur kosong (`continue`), perbaiki/tolak nilai negatif, hitung data valid, laporkan alasan penolakan.

In [ ]:
data_valid = []
alasan = {"bmi_kosong": 0, "smoking_unknown": 0, "gender_other": 0, "bmi_negatif": 0}

for _, baris in df.iterrows():
    if pd.isna(baris["bmi"]):
        alasan["bmi_kosong"] += 1
        continue
    if baris["smoking_status"] == "Unknown":
        alasan["smoking_unknown"] += 1
        continue
    if baris["gender"] == "Other":
        alasan["gender_other"] += 1
        continue
    if baris["bmi"] < 0:  # tidak ada di dataset ini, cabang disiapkan ikut slide
        alasan["bmi_negatif"] += 1
        continue
    data_valid.append(baris)

n_valid = len(data_valid)
n_tolak = sum(alasan.values())
print(f"valid: {n_valid} ({n_valid/len(df)*100:.1f}%)")
print(f"ditolak: {n_tolak} ({n_tolak/len(df)*100:.1f}%)")
print(alasan)

# ponytail: tiap baris tepat satu nasib (valid atau satu alasan), maksimal cek tumpang tindih
assert n_valid + n_tolak == len(df)
assert alasan["bmi_negatif"] == 0  # dokumentasi: dataset ini memang tanpa bmi negatif

**Analisis:** penolakan didominasi `smoking_status Unknown` (sekitar 30%), lalu `bmi` kosong (sekitar 4%), dan satu baris `gender Other`. Tidak ada `bmi` negatif, cabang itu dipertahankan sebagai pengaman mengikuti slide. Urutan cek memengaruhi angka per alasan (baris yang dobel masalah tercatat pada alasan pertama), tetapi total valid + ditolak selalu sama dengan 5.110. Untuk proyek akhir, baris Unknown dan bmi kosong tidak dibuang melainkan ditangani di preprocessing (notebook `04`); penyaringan keras di sini hanya demonstrasi kontrol alur, bukan keputusan final pipeline.

## 10. Refleksi dan ringkasan

| Butuh apa? | Pakai apa? |
|---|---|
| Keputusan | `if-elif-else`, batas ketat dulu |
| Telusuri koleksi | `for` + `enumerate`/`zip` |
| Berhenti ikut kondisi | `while`, pengendali selalu maju |
| Abaikan satu item | `continue` |
| Hentikan semua | `break` (darurat saja) |

**Kesimpulan:** kontrol alur adalah fondasi pipeline data. Kondisi membuat program adaptif, loop mengotomatisasi proses berulang, dan `break`/`continue` menangani kasus khusus. Kualitas data dimulai dari logika yang jelas. Pertemuan berikutnya mengemas logika ini menjadi fungsi dan modul.

**Bahan screenshot laporan buku panduan:** tiap nomor bagian di atas adalah satu tahapan (1 muat data, 2 boolean, 3 percabangan, 4 for, 5 while, 6 break/continue, 7 nested loop, 8 comprehension, 9 penyaringan). Screenshot kode + output + paragraf analisis di bawahnya.